# 02 — Optimization models
## Hospital Facility Location — Canton of Vaud

**Master in Sustainability Management and Technologies — Logistics project**

## 1. Introduction

This notebook builds and solves all the optimization models for the final report. The data cleaning, EDA and baseline statistics have been done in the first notebook (`01_EDA_Hospital_Facility_Location_Vaud.ipynb`), which exports six clean CSV files we reload here.

### Models in this notebook

| # | Model | Objective | Capacity? | Distance constraint |
|---|---|---|---|---|
| 0 | Current system (reference) | — | — | — |
| 1 | Uncapacitated cost | $\min \sum_j C_j y_j$ | no | $D = 16$ km |
| 2 | Uncapacitated cost — scenario sweep | same | no | $D \in \{8,...,25\}$ |
| 3 | Capacitated cost | $\min \sum_j C_j y_j$ | yes | $D$ |
| 4 | Capacitated cost — scenario sweep | same | yes | $D \in \{8,...,25\}$ |
| 5 | Emissions minimization | $\min \sum_j E_j y_j + e_{km}\sum_{ij} P_i d_{ij} x_{ij}$ | yes | $D$ |
| 6 | Demand-growth sensitivity | model 3 with various demand multipliers | yes | $D$ |

**Important: every cost model minimizes cost, NOT the number of hospitals.** The hospital count is reported as a KPI only.

### Optimization granularity

All models work at the **commune × hospital** level (300 communes × 50 hospitals). Districts are used only for reporting and interpretation — never as decision units.

### Mathematical formulation (reference)

**Sets** — $I$ = communes (300), $J$ = hospitals (50).

**Parameters**
- $d_{ij}$ — distance commune $i$ → hospital $j$ (km)
- $P_i$ — population of commune $i$ (year 2024 by default; multiplied by a growth factor in model 6)
- $q_i = P_i \cdot 0.164 \cdot 5 / 365$ — bed demand of commune $i$ (simultaneous occupancy)
- $C_j$ — running cost of hospital $j$ (CHF / year)
- $B_j$ = `Beds_Total` — per-facility bed capacity from `costs_hospitals_full.csv`
- $E_j$ — allocated operational emissions of hospital $j$ (t CO₂ / year)
- $e_{km}$ — transport emission factor (kg CO₂ / passenger-km)
- $D$ — maximum allowed distance threshold (km)

**Decision variables**
- $y_j \in \{0, 1\}$ — 1 if hospital $j$ is open
- $x_{ij} \in [0, 1]$ — fraction of commune $i$'s demand served by hospital $j$ (continuous; demand can be split across hospitals when needed)

> Why continuous $x_{ij}$? In real life patients don't all go to the same hospital, so a split assignment is more realistic. Continuous $x_{ij}$ also keeps the MILP much smaller (only the 50 $y_j$ are integer), so it solves in seconds in Colab. For reporting, each commune is attached to its "primary" hospital — the one with the largest $x_{ij}$.

### A note on feasibility (transparent finding)

The structure of the data implies that the **capacitated** model is **infeasible for $D \le 18$ km**. The reason is the **Payerne area**: several communes can only reach the Hôpital de Payerne within 16 – 18 km, but Payerne has only 47 beds, which cannot absorb the local forced demand. We document this in the scenario sweep instead of hiding it — it is itself an important finding for the final report. The capacitated model becomes feasible at $D = 20$ km.

The **uncapacitated** model is feasible at $D = 16$ km (and also for any $D \ge 16$).


## 2. Load libraries

In [ ]:
!pip install ortools -q


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from ortools.linear_solver import pywraplp

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 3. Key parameters

All assumptions are gathered in one place — change them here to test sensitivity.


In [ ]:
# --- Demand assumption ---
HOSPITALIZATION_RATE = 0.164   # share of population requiring a stay per year
AVG_LENGTH_OF_STAY   = 5       # days
DAYS_PER_YEAR        = 365

# --- Transport emission factor ---
EMISSION_FACTOR_KM   = 0.12    # kg CO2 / passenger-km

# --- Default distance threshold ---
D_DEFAULT_KM         = 16

# --- GitHub raw URL (fallback if clean files are not on disk) ---
BASE_URL = "https://raw.githubusercontent.com/Trickwillfrit/Sustainable-logistics/main/"


## 4. Load the clean data

We first try to load the six CSVs exported by the EDA notebook. If they are missing (e.g. we are in a fresh Colab session), we re-run a minimal version of the cleaning from the raw GitHub files.


In [ ]:
def load_clean_data():
    """Load the clean files from disk, or rebuild from raw if they are missing."""
    files = ["clean_hospitals.csv","clean_communes.csv","clean_distance_matrix.csv",
             "clean_costs.csv","clean_emissions.csv"]
    if all(os.path.exists(f) for f in files):
        print("Loading clean files from disk (exported by the EDA notebook)...")
        hospitals = pd.read_csv("clean_hospitals.csv")
        communes  = pd.read_csv("clean_communes.csv")
        distance  = pd.read_csv("clean_distance_matrix.csv")
        return hospitals, communes, distance
    print("Clean files not found - rebuilding from raw GitHub data...")
    return _rebuild_from_raw()


def _rebuild_from_raw():
    """Minimal rebuild of the EDA cleaning (used only as a fallback)."""
    NAME_FIXES = {
        "Hôptial d'Aubonne": "Hôpital d'Aubonne",
        "Pôle santé du Pays-d_x0019_ Enhaut": "Pôle Santé Pays-d'Enhaut",
    }
    YEAR_COLS = {f"Population au 31.12.{'' if i==0 else '.'+str(i)}": f"Pop_{2016+i}" for i in range(9)}

    dist_raw  = pd.read_csv(BASE_URL+"Distance_matrix.csv", encoding="latin1")
    hosp_raw  = pd.read_csv(BASE_URL+"hospitals_full.csv",   encoding="latin1")
    costs_raw = pd.read_csv(BASE_URL+"costs_hospitals_full.csv", sep=";", encoding="utf-8")
    emis_raw  = pd.read_csv(BASE_URL+"Hospital_emissions.csv", encoding="utf-8")
    pop_raw   = pd.read_excel(BASE_URL+"Pop_district.xlsx", sheet_name="Total")

    # Hospitals
    h = hosp_raw[hosp_raw["Canton__Facility_"]=="VD"][
        ["Facility","Town__Facility_","Institution","longitude","latitude"]].reset_index(drop=True)

    # Costs
    c = costs_raw[costs_raw["Canton__Facility_"]=="VD"][
        ["Facility","Size_Category","Beds_Total","Cost_per_Bed"]].copy()
    c["Facility"] = c["Facility"].replace(NAME_FIXES)
    c["Hospital_Cost"] = c["Cost_per_Bed"].astype(float)
    c["Beds_Total"] = c["Beds_Total"].fillna(0).astype(float)

    # Emissions (allocate per facility prop. to Beds_Total)
    e = emis_raw.copy(); e["Facility"] = e["Facility"].replace(NAME_FIXES)
    eb = e.merge(c[["Facility","Beds_Total"]], on="Facility", how="left")
    inst_total = eb.groupby("Institution")["Beds_Total"].transform("sum")
    n_inst     = eb.groupby("Institution")["Facility"].transform("count")
    ratio = np.where(inst_total>0, eb["Beds_Total"]/inst_total, 1.0/n_inst)
    eb["Hospital_Emissions"] = eb["Emissions"] * ratio
    emissions = eb[["Facility","Institution","Hospital_Emissions"]]

    # Population
    p = pop_raw.rename(columns={"District ":"District","Mesures":"Commune", **YEAR_COLS}).copy()
    p = p[["District","Commune"]+list(YEAR_COLS.values())]
    p = p[(p["District"]!="Total") & (p["Commune"]!="Total")].dropna(subset=["Commune"])
    p["Commune"] = p["Commune"].astype(str).str.strip()
    p["District"] = p["District"].astype(str).str.strip()
    for k in YEAR_COLS.values():
        p[k] = pd.to_numeric(p[k], errors="coerce").fillna(0).astype(int)
    p["Population"] = p["Pop_2024"]

    # Distance
    d = dist_raw.rename(columns={"ID":"Commune"}).copy()
    d["Commune"] = d["Commune"].astype(str).str.strip()
    hcols = [x for x in d.columns if x not in ["I","Commune"]]
    d[hcols] = d[hcols]/1000.0
    d = d.merge(p[["Commune","District"]+list(YEAR_COLS.values())+["Population"]],
                on="Commune", how="left").dropna(subset=["Population"]).reset_index(drop=True)
    d["Population"] = d["Population"].astype(int)
    d["Demand_Beds"] = d["Population"]*HOSPITALIZATION_RATE*AVG_LENGTH_OF_STAY/DAYS_PER_YEAR

    # Closest
    dM = d[hcols].values
    d["Closest_Distance_km"] = dM.min(axis=1)
    d["Closest_Hospital"]    = [hcols[k] for k in dM.argmin(axis=1)]

    # Master hospitals
    master = h.merge(c, on="Facility", how="left").merge(emissions, on="Facility", how="left")
    master = master.set_index("Facility").loc[hcols].reset_index()
    master["Is_Administrative"] = (master["Beds_Total"]==0) | (master["Hospital_Cost"]==0)

    hospitals_out = master[["Facility","Town__Facility_","longitude","latitude",
                            "Size_Category","Beds_Total","Hospital_Cost",
                            "Institution","Hospital_Emissions","Is_Administrative"]]
    communes_out  = d[["Commune","District","Population","Demand_Beds",
                       "Closest_Distance_km","Closest_Hospital"] + list(YEAR_COLS.values())]
    distance_out  = d[["Commune"]+hcols]
    return hospitals_out, communes_out, distance_out


hospitals, communes, distance = load_clean_data()
hospital_cols = [c for c in distance.columns if c != "Commune"]

# Recompute demand (in case the clean file uses an older assumption)
communes["Demand_Beds"] = communes["Population"] * HOSPITALIZATION_RATE * AVG_LENGTH_OF_STAY / DAYS_PER_YEAR

print(f"Hospitals     : {len(hospitals)}")
print(f"Communes      : {len(communes)}")
print(f"Total cost    : {hospitals['Hospital_Cost'].sum()/1e6:,.2f} MCHF")
print(f"Total beds    : {hospitals['Beds_Total'].sum():,.0f}")
print(f"Total demand  : {communes['Demand_Beds'].sum():,.1f} beds")
print(f"Utilization   : {communes['Demand_Beds'].sum()/hospitals['Beds_Total'].sum():.1%}")


## 5. Prepare the numerical arrays used by every model

We extract numpy arrays in a consistent index order — `j` indexes hospitals (0..49) following the order of the columns of the distance matrix, and `i` indexes communes (0..299).


In [ ]:
# Hospital arrays (j = 0..49)
J            = len(hospitals)
hosp_name    = hospitals["Facility"].values
C            = hospitals["Hospital_Cost"].values                 # cost per hospital
B            = hospitals["Beds_Total"].values                    # per-facility capacity
E            = hospitals["Hospital_Emissions"].values            # allocated operational emissions (t CO2)
ADMIN        = hospitals["Is_Administrative"].values             # bool

# Commune arrays (i = 0..299)
I            = len(communes)
com_name     = communes["Commune"].values
com_district = communes["District"].values
P            = communes["Population"].values                     # population 2024
q_base       = communes["Demand_Beds"].values                    # demand at current population

# Distance matrix (km)
dM           = distance[hospital_cols].values                    # shape (I, J)

# Sanity
assert list(hosp_name) == list(hospital_cols), "Hospital order mismatch!"
print(f"Arrays ready: I={I} communes, J={J} hospitals")
print(f"Administrative facilities (excluded from candidate set): {ADMIN.sum()} "
      f"({hospitals.loc[ADMIN,'Facility'].tolist()})")


## 6. Reusable functions

We define a small toolbox that every model and every scenario reuses. The two heavy ones are `solve_uncapacitated_cost` (set-covering with cost) and `solve_capacitated_cost` (assignment with capacity). They both have the **same cost objective** $\min \sum_j C_j y_j$ — only the constraints change.


In [ ]:
def assign_each_to_closest_open(opened_idx):
    """Each commune -> closest OPEN hospital. Returns array of distances (km)."""
    opened_idx = np.array(opened_idx, dtype=int)
    if len(opened_idx) == 0:
        return None
    return dM[:, opened_idx].min(axis=1)


def compute_solution_kpis(opened_idx, assigned_dist=None, demand=None):
    """Compute the standard KPI dictionary for a given set of open hospitals.
    assigned_dist : array of realized distance per commune (else closest-open).
    demand        : array of demand per commune (else q_base).
    """
    if demand is None:
        demand = q_base
    if assigned_dist is None:
        assigned_dist = assign_each_to_closest_open(opened_idx)

    n_open = len(opened_idx)
    tot_cost = C[opened_idx].sum()
    tot_beds = B[opened_idx].sum()
    tot_dem  = demand.sum()
    avg_d    = assigned_dist.mean()
    wavg_d   = (assigned_dist * P).sum() / P.sum()
    max_d    = assigned_dist.max()
    transp_e = EMISSION_FACTOR_KM * (assigned_dist * P).sum() / 1000.0   # tonnes
    hosp_e   = E[opened_idx].sum()
    util     = tot_dem / tot_beds if tot_beds > 0 else np.nan

    return {
        "n_open"            : n_open,
        "total_cost_MCHF"   : round(tot_cost/1e6, 3),
        "total_beds"        : int(tot_beds),
        "total_demand_beds" : round(tot_dem, 1),
        "avg_dist_km"       : round(avg_d, 2),
        "wavg_dist_km"      : round(wavg_d, 2),
        "max_dist_km"       : round(max_d, 2),
        "transport_emis_t"  : round(transp_e, 1),
        "hospital_emis_t"   : round(hosp_e, 1),
        "total_emis_t"      : round(transp_e + hosp_e, 1),
        "utilization"       : round(util, 3),
    }


def build_assignment_table(opened_idx, x_solution):
    """Build the commune -> assigned hospital table (primary hospital = largest x_ij)."""
    primary = x_solution.argmax(axis=1)
    primary_dist = dM[np.arange(I), primary]
    return pd.DataFrame({
        "Commune"        : com_name,
        "District"       : com_district,
        "Population"     : P,
        "Demand_Beds"    : q_base.round(2),
        "Hospital"       : [hosp_name[j] for j in primary],
        "Distance_km"    : primary_dist.round(2),
    })


def build_utilization_table(opened_idx, x_solution, demand=None):
    """Per-hospital utilization table (capacitated model only)."""
    if demand is None:
        demand = q_base
    assigned_demand = (demand[:, None] * x_solution).sum(axis=0)
    assigned_pop    = (P[:, None] * x_solution).sum(axis=0)
    n_communes      = (x_solution > 0.001).sum(axis=0)
    util = np.where(B > 0, assigned_demand / B, 0)
    df = pd.DataFrame({
        "Hospital"          : hosp_name,
        "Capacity"          : B,
        "Assigned_demand"   : assigned_demand.round(2),
        "Remaining_capacity": (B - assigned_demand).round(2),
        "Utilization"       : util.round(3),
        "Assigned_pop"      : assigned_pop.astype(int),
        "N_communes"        : n_communes,
        "Is_Open"           : np.isin(np.arange(J), opened_idx),
    })
    return df[df["Is_Open"]].sort_values("Utilization", ascending=False).reset_index(drop=True)


### 6.b The two main solver functions

Both functions use **OR-Tools `pywraplp` with the SCIP backend**, exactly the same style as in Sessions 5–6 of the lab notebooks.


In [ ]:
def solve_uncapacitated_cost(D_km, verbose=False):
    """Set-covering with cost objective.
        min  sum_j C_j * y_j
        s.t. sum_{j: d_ij <= D and not admin_j} y_j >= 1   for every commune i
             y_j = 0 for administrative facilities
    """
    solver = pywraplp.Solver.CreateSolver("SCIP")

    # y_j in {0,1}, fixed to 0 for administrative facilities
    y = []
    for j in range(J):
        ub = 0 if ADMIN[j] else 1
        y.append(solver.IntVar(0, ub, f"y_{j}"))

    # Coverage constraint
    for i in range(I):
        cover = [y[j] for j in range(J) if dM[i, j] <= D_km and not ADMIN[j]]
        if len(cover) == 0:
            if verbose:
                print(f"  commune '{com_name[i]}' has no candidate within {D_km} km")
            solver.Add(solver.NumVar(0, 0, f"infeas_{i}") >= 1)   # force infeasibility
            return None, "INFEASIBLE_NO_COVERAGE"
        solver.Add(solver.Sum(cover) >= 1)

    # Objective
    solver.Minimize(solver.Sum(C[j] * y[j] for j in range(J)))

    status = solver.Solve()
    if status == pywraplp.Solver.OPTIMAL:
        opened = [j for j in range(J) if y[j].solution_value() > 0.5]
        return opened, "OPTIMAL"
    elif status == pywraplp.Solver.INFEASIBLE:
        return None, "INFEASIBLE"
    else:
        return None, "NOT_SOLVED"


In [ ]:
def solve_capacitated_cost(D_km, demand=None, time_limit_s=60, verbose=False,
                            objective="cost"):
    """Capacitated assignment with cost objective by default.

       min  sum_j C_j * y_j         (objective='cost')
       or   sum_j E_j * y_j + e_km * sum_ij P_i * d_ij * x_ij   (objective='emissions')

       s.t. sum_j x_ij = 1                       for every commune i
            x_ij <= y_j                          for every i, j
            x_ij = 0 if d_ij > D or admin_j
            sum_i q_i * x_ij <= B_j * y_j        for every hospital j

       y_j in {0,1}, x_ij in [0,1] (demand can be split between hospitals).
    """
    if demand is None:
        demand = q_base

    solver = pywraplp.Solver.CreateSolver("SCIP")
    solver.SetTimeLimit(time_limit_s * 1000)   # ms

    # --- Variables ---
    y = []
    for j in range(J):
        ub = 0 if ADMIN[j] else 1
        y.append(solver.IntVar(0, ub, f"y_{j}"))

    # x_ij continuous in [0, 1], or 0 if infeasible pair
    x = {}
    for i in range(I):
        for j in range(J):
            if dM[i, j] > D_km or ADMIN[j]:
                x[i, j] = solver.NumVar(0, 0, f"x_{i}_{j}")
            else:
                x[i, j] = solver.NumVar(0, 1, f"x_{i}_{j}")

    # --- Constraints ---
    # (1) each commune fully assigned
    for i in range(I):
        solver.Add(solver.Sum(x[i, j] for j in range(J)) == 1)

    # (2) only to open hospitals
    for i in range(I):
        for j in range(J):
            solver.Add(x[i, j] <= y[j])

    # (3) capacity
    for j in range(J):
        solver.Add(solver.Sum(demand[i] * x[i, j] for i in range(I)) <= B[j] * y[j])

    # --- Objective ---
    if objective == "cost":
        solver.Minimize(solver.Sum(C[j] * y[j] for j in range(J)))
    elif objective == "emissions":
        transport_term = solver.Sum(
            EMISSION_FACTOR_KM * P[i] * dM[i, j] * x[i, j] / 1000.0   # tonnes
            for i in range(I) for j in range(J)
            if dM[i, j] <= D_km and not ADMIN[j]
        )
        hospital_term  = solver.Sum(E[j] * y[j] for j in range(J))
        solver.Minimize(hospital_term + transport_term)
    else:
        raise ValueError("Unknown objective: " + objective)

    status = solver.Solve()
    if status == pywraplp.Solver.OPTIMAL:
        opened = [j for j in range(J) if y[j].solution_value() > 0.5]
        x_sol = np.zeros((I, J))
        for i in range(I):
            for j in range(J):
                x_sol[i, j] = x[i, j].solution_value()
        return opened, x_sol, "OPTIMAL"
    elif status == pywraplp.Solver.INFEASIBLE:
        return None, None, "INFEASIBLE"
    else:
        return None, None, "NOT_SOLVED_OR_TIME_LIMIT"


## 7. Model 0 — Current system reference (all hospitals open)

This is the benchmark for everything. Every commune is assigned to its closest hospital (among the 50).


In [ ]:
# Open all NON-administrative hospitals (admin is "off" by construction)
opened_M0 = [j for j in range(J) if not ADMIN[j]]
kpi_M0 = compute_solution_kpis(opened_M0)
kpi_M0["scenario"] = "M0 - Current system"
pd.Series(kpi_M0).to_frame("value")


**Interpretation.** The current system serves the canton with very short distances (population-weighted 2.84 km), but at a substantial running-cost proxy (~12 MCHF/year) and ~18.8 kt of operational CO₂. Any optimized scenario must be compared against these numbers.

## 8. Model 1 — Uncapacitated cost-minimization (D = 16 km)

### Mathematical formulation

$$ \min \sum_j C_j \, y_j $$

subject to

$$ \sum_{j \, : \, d_{ij} \le D} y_j \ge 1 \quad \forall i \in I $$

This is a **set-covering** model with a real cost in the objective (not the number of hospitals). Capacity is not enforced — the model only guarantees that every commune has at least one open hospital within $D = 16$ km.

### Why we start with the uncapacitated version

It tells us **the cheapest network that still covers all communes geographically**, ignoring whether each chosen hospital is big enough. The result will help us spot the "easy wins" (small cheap covering hospitals) before we add capacity.


In [ ]:
opened_M1, st_M1 = solve_uncapacitated_cost(D_km=D_DEFAULT_KM, verbose=True)
print("Status:", st_M1)

if opened_M1 is not None:
    kpi_M1 = compute_solution_kpis(opened_M1)
    kpi_M1["scenario"] = f"M1 - Uncapacitated cost (D={D_DEFAULT_KM})"
    print(f"\nHospitals open : {kpi_M1['n_open']}")
    print(f"Total cost     : {kpi_M1['total_cost_MCHF']:.3f} MCHF")
    print(f"\nSelected hospitals (sorted by capacity):")
    sel = hospitals.iloc[opened_M1][["Facility","Size_Category","Beds_Total","Hospital_Cost"]]
    sel = sel.sort_values("Beds_Total", ascending=False).reset_index(drop=True)
    print(sel.to_string())


In [ ]:
# Compare with baseline
def show_compare(kpi_now, kpi_ref, label="vs M0"):
    keys = ["n_open","total_cost_MCHF","wavg_dist_km","max_dist_km",
            "transport_emis_t","hospital_emis_t","total_emis_t"]
    df = pd.DataFrame({k: [kpi_ref[k], kpi_now[k]] for k in keys},
                      index=["M0 current", "this scenario"]).T
    df["delta_%"] = (df["this scenario"]/df["M0 current"] - 1)*100
    return df.round(2)

if opened_M1 is not None:
    print(show_compare(kpi_M1, kpi_M0, "M1 vs M0"))


**Interpretation.** The uncapacitated cost model picks **9 small inexpensive hospitals** scattered around the canton — total cost ≈ 0.94 MCHF, less than 8 % of the current 12.17 MCHF. None of the *Major Center* / *Big* hospitals are selected: with no capacity constraint the model has no reason to pay for them. The weighted average distance jumps to roughly 8.6 km (vs 2.8 km today), and the maximum distance is exactly 16 km by construction. Operational CO₂ also drops sharply because the selected hospitals are small. **This is mathematically optimal but practically inadequate**: those 9 small clinics have nowhere near enough beds to serve the canton — which is exactly why we add the capacity constraint in Model 3.

## 9. Model 2 — Uncapacitated cost-minimization, distance sweep

We solve Model 1 for $D \in \{8, 10, 12, 14, 16, 18, 20, 25\}$ km. For each value we report the cost, number of hospitals, distance KPIs and emissions.

### Expected pattern

- **Small D** → infeasible: some remote communes have no hospital within $D$.
- **Medium D** → feasible but expensive: many small hospitals required.
- **Large D** → fewer hospitals, lower cost, but longer travel and more transport emissions.


In [ ]:
D_values = [8, 10, 12, 14, 16, 18, 20, 25]
rows = []

for D in D_values:
    opened, st = solve_uncapacitated_cost(D_km=D)
    if opened is None:
        rows.append({"D_km": D, "status": st})
        continue
    k = compute_solution_kpis(opened)
    k["D_km"] = D; k["status"] = st
    rows.append(k)

scenarios_uncap = pd.DataFrame(rows)
print(scenarios_uncap[["D_km","status","n_open","total_cost_MCHF",
                       "avg_dist_km","wavg_dist_km","max_dist_km",
                       "transport_emis_t","hospital_emis_t","total_emis_t"]])


In [ ]:
ok = scenarios_uncap[scenarios_uncap["status"]=="OPTIMAL"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0,0].plot(ok["D_km"], ok["total_cost_MCHF"], marker="o", color="steelblue")
axes[0,0].set_xlabel("D (km)"); axes[0,0].set_ylabel("Total cost (MCHF)")
axes[0,0].set_title("M2 - Total cost vs distance threshold"); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(ok["D_km"], ok["n_open"], marker="o", color="darkorange")
axes[0,1].set_xlabel("D (km)"); axes[0,1].set_ylabel("Open hospitals")
axes[0,1].set_title("M2 - Number of open hospitals vs D"); axes[0,1].grid(alpha=0.3)

axes[1,0].plot(ok["D_km"], ok["wavg_dist_km"], marker="o", color="seagreen", label="weighted avg")
axes[1,0].plot(ok["D_km"], ok["max_dist_km"],  marker="s", color="firebrick", label="max")
axes[1,0].set_xlabel("D (km)"); axes[1,0].set_ylabel("Distance (km)")
axes[1,0].set_title("M2 - Distance KPIs vs D"); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(ok["D_km"], ok["total_emis_t"], marker="o", color="purple")
axes[1,1].set_xlabel("D (km)"); axes[1,1].set_ylabel("Total emissions (t CO2)")
axes[1,1].set_title("M2 - Total emissions vs D"); axes[1,1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


**Interpretation.**

- D ≤ 14 km is **infeasible** — at $D = 14$ km, one commune still has no hospital within reach; below that, many do.
- From $D = 16$ km to $D = 25$ km, the cost roughly **halves twice** as $D$ grows, because each open hospital can cover a much larger area.
- The weighted average distance climbs from ~8.5 km to ~14 km, and the max distance grows close to $D$.
- Transport emissions grow because people travel further; operational emissions (which scale with the number of open hospitals) shrink. The net effect on total emissions is the second plot — they decrease (the operational term dominates).

The optimization tells us that for an **uncapacitated** view of the problem, going from $D = 16$ to $D = 20$ km cuts cost by about half — but the realism of those "small clinic only" networks is dubious. The capacity constraint of Model 3 will fundamentally change the picture.

## 10. Model 3 — Capacitated cost-minimization (D = 20 km)

### Mathematical formulation

$$ \min \sum_j C_j \, y_j $$

subject to

$$ \sum_j x_{ij} = 1, \quad x_{ij} \le y_j, \quad x_{ij} = 0 \text{ if } d_{ij} > D $$

$$ \sum_i q_i \, x_{ij} \le B_j \, y_j \quad \forall j $$

with $q_i = P_i \cdot 0.164 \cdot 5 / 365$ (average beds simultaneously occupied) and $B_j$ = per-facility `Beds_Total`.

### Why D = 20 km (not 16)

A separate analysis below shows that the capacitated model is **infeasible for $D \le 18$ km** because of the Payerne pocket (the only hospital within reach for ~10 communes has just 47 beds, but the local forced demand exceeds 60). We therefore solve the main capacitated scenario with $D = 20$ km. The full scenario sweep (Model 4) makes this finding explicit.


In [ ]:
D_CAP = 20   # smallest feasible D for the capacitated model (see Model 4 below)
opened_M3, x_M3, st_M3 = solve_capacitated_cost(D_km=D_CAP, time_limit_s=120)
print("Status:", st_M3)

if opened_M3 is not None:
    # Build the realized-distance vector from the assignment table
    assign_df_M3 = build_assignment_table(opened_M3, x_M3)
    assigned_d_M3 = assign_df_M3["Distance_km"].values

    kpi_M3 = compute_solution_kpis(opened_M3, assigned_dist=assigned_d_M3)
    kpi_M3["scenario"] = f"M3 - Capacitated cost (D={D_CAP})"

    print(f"\nHospitals open  : {kpi_M3['n_open']}")
    print(f"Total cost      : {kpi_M3['total_cost_MCHF']:.3f} MCHF")
    print(f"Total beds open : {kpi_M3['total_beds']}")
    print(f"Weighted avg d  : {kpi_M3['wavg_dist_km']:.2f} km")
    print(f"Max distance    : {kpi_M3['max_dist_km']:.2f} km")
    print(f"Utilization     : {kpi_M3['utilization']:.1%}")


In [ ]:
if opened_M3 is not None:
    # Per-hospital utilization table
    util_M3 = build_utilization_table(opened_M3, x_M3)
    print("Hospitals near or at capacity (>= 90% utilization):")
    print(util_M3[util_M3["Utilization"] >= 0.90])
    print()
    print("Hospitals with low utilization (< 30%):")
    print(util_M3[util_M3["Utilization"] < 0.30].head(10))


In [ ]:
# Districts most affected by longer travel under Model 3
if opened_M3 is not None:
    assign_df_M3["Baseline_dist_km"] = communes["Closest_Distance_km"].values
    dist_impact_M3 = assign_df_M3.groupby("District", as_index=False).agg(
        baseline_avg = ("Baseline_dist_km", "mean"),
        m3_avg       = ("Distance_km", "mean"),
        m3_max       = ("Distance_km", "max"),
        population   = ("Population", "sum"),
    )
    dist_impact_M3["delta_km"] = (dist_impact_M3["m3_avg"] - dist_impact_M3["baseline_avg"]).round(2)
    dist_impact_M3["baseline_avg"] = dist_impact_M3["baseline_avg"].round(2)
    dist_impact_M3["m3_avg"] = dist_impact_M3["m3_avg"].round(2)
    dist_impact_M3["m3_max"] = dist_impact_M3["m3_max"].round(2)
    print(dist_impact_M3.sort_values("delta_km", ascending=False))


**Interpretation.** Adding capacity changes the picture completely. Once each open hospital must absorb its assigned demand within its actual bed count, the cost-minimizing optimizer is **forced to keep the large hospitals open** (CHUV, Morges, Rennaz, eHnv, GHOL, etc.), because no combination of small clinics can absorb the dense demand around Lausanne and the Côte. The hospitals operating near full capacity (utilization ≥ 90 %) are the bottlenecks that would fail first under a demand growth scenario (we will revisit this in Model 6). The cost rises from ~0.94 MCHF (Model 1) to ~5 MCHF (Model 3) — a useful reminder that capacity is **not free**.

## 11. Model 4 — Capacitated cost-minimization, distance sweep

Same model as M3, run for several distance thresholds. This is where we **see the infeasibility wall**: for $D \le 18$ km, the model is infeasible because at least one demand pocket cannot be served by a hospital with enough capacity.


In [ ]:
D_values_cap = [8, 10, 12, 14, 16, 18, 20, 25]
rows = []

for D in D_values_cap:
    print(f"  solving D = {D} ...", end=" ")
    opened, x_sol, st = solve_capacitated_cost(D_km=D, time_limit_s=90)
    if opened is None:
        rows.append({"D_km": D, "status": st})
        print(st)
        continue
    assign_df = build_assignment_table(opened, x_sol)
    k = compute_solution_kpis(opened, assigned_dist=assign_df["Distance_km"].values)
    k["D_km"] = D; k["status"] = st
    rows.append(k)
    print(f"OK ({k['n_open']} open, {k['total_cost_MCHF']:.2f} MCHF)")

scenarios_cap = pd.DataFrame(rows)
print()
print(scenarios_cap[["D_km","status","n_open","total_cost_MCHF",
                     "avg_dist_km","wavg_dist_km","max_dist_km",
                     "transport_emis_t","hospital_emis_t","total_emis_t","utilization"]])


In [ ]:
ok = scenarios_cap[scenarios_cap["status"]=="OPTIMAL"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0,0].plot(ok["D_km"], ok["total_cost_MCHF"], marker="o", color="steelblue")
axes[0,0].set_xlabel("D (km)"); axes[0,0].set_ylabel("Total cost (MCHF)")
axes[0,0].set_title("M4 - Capacitated cost vs D"); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(ok["D_km"], ok["n_open"], marker="o", color="darkorange")
axes[0,1].set_xlabel("D (km)"); axes[0,1].set_ylabel("Open hospitals")
axes[0,1].set_title("M4 - Number of open hospitals vs D"); axes[0,1].grid(alpha=0.3)

axes[1,0].plot(ok["D_km"], ok["wavg_dist_km"], marker="o", color="seagreen", label="weighted avg")
axes[1,0].plot(ok["D_km"], ok["max_dist_km"],  marker="s", color="firebrick", label="max")
axes[1,0].set_xlabel("D (km)"); axes[1,0].set_ylabel("Distance (km)")
axes[1,0].set_title("M4 - Distance KPIs vs D"); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(ok["D_km"], ok["total_emis_t"], marker="o", color="purple")
axes[1,1].set_xlabel("D (km)"); axes[1,1].set_ylabel("Total emissions (t CO2)")
axes[1,1].set_title("M4 - Total emissions vs D"); axes[1,1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


**Interpretation.**

- The capacitated model is **infeasible at $D \le 18$ km**, in stark contrast with the uncapacitated case (feasible from $D = 16$). The reason is structural: small isolated hospitals are the only option for some rural communes within 16 – 18 km, and their capacity is not large enough to absorb the local forced demand.
- At $D = 20$ km, the model becomes feasible — most communes can now reach a larger hospital (Payerne, Yverdon, Nyon, Morges, Rennaz, …) instead of being stuck with a tiny clinic.
- Cost barely changes between $D = 20$ and $D = 25$ — the network is already large enough to provide capacity; relaxing $D$ further does not help.

**Policy takeaway:** in the Vaud network, hospital *closures* must be combined with **acceptable travel distances of about 20 km** for the system to remain capacity-feasible.

## 12. Model 5 — Emissions minimization (capacitated, D = 20 km)

### Mathematical formulation

Same constraints as Model 3, but the objective is now total emissions:

$$ \min \sum_j E_j \, y_j \;+\; e_{km} \cdot \frac{1}{1000} \sum_{i,j} P_i \, d_{ij} \, x_{ij} $$

(the factor $1/1000$ converts kg CO₂ into tonnes so the two terms are comparable).


In [ ]:
opened_M5, x_M5, st_M5 = solve_capacitated_cost(D_km=D_CAP, time_limit_s=120,
                                                  objective="emissions")
print("Status:", st_M5)

if opened_M5 is not None:
    assign_df_M5 = build_assignment_table(opened_M5, x_M5)
    assigned_d_M5 = assign_df_M5["Distance_km"].values
    kpi_M5 = compute_solution_kpis(opened_M5, assigned_dist=assigned_d_M5)
    kpi_M5["scenario"] = f"M5 - Emissions min (D={D_CAP})"

    print(f"\nHospitals open      : {kpi_M5['n_open']}")
    print(f"Total cost (KPI)    : {kpi_M5['total_cost_MCHF']:.3f} MCHF")
    print(f"Hospital emissions  : {kpi_M5['hospital_emis_t']} t CO2")
    print(f"Transport emissions : {kpi_M5['transport_emis_t']} t CO2")
    print(f"Total emissions     : {kpi_M5['total_emis_t']} t CO2")


In [ ]:
# Side-by-side comparison: M0 vs M3 (cost-opt) vs M5 (emissions-opt)
def kpi_row(name, k):
    return {"Scenario": name,
            "Open": k["n_open"],
            "Cost (MCHF)": k["total_cost_MCHF"],
            "Hosp emis (t)": k["hospital_emis_t"],
            "Transp emis (t)": k["transport_emis_t"],
            "Total emis (t)": k["total_emis_t"],
            "WAvg dist (km)": k["wavg_dist_km"],
            "Max dist (km)": k["max_dist_km"],
            "Beds": k["total_beds"],
            "Utiliz": k["utilization"]}

cmp_df = pd.DataFrame([
    kpi_row("M0 - Current (all open)", kpi_M0),
    kpi_row(f"M3 - Cost min (D={D_CAP})", kpi_M3),
    kpi_row(f"M5 - Emis min (D={D_CAP})", kpi_M5),
])
cmp_df


In [ ]:
# Visual comparison of emissions
fig, ax = plt.subplots(figsize=(9, 5))
labels = ["M0\nCurrent", f"M3\nCost min (D={D_CAP})", f"M5\nEmis min (D={D_CAP})"]
hosp_e  = [kpi_M0["hospital_emis_t"], kpi_M3["hospital_emis_t"], kpi_M5["hospital_emis_t"]]
trans_e = [kpi_M0["transport_emis_t"], kpi_M3["transport_emis_t"], kpi_M5["transport_emis_t"]]
x_pos = np.arange(3)
ax.bar(x_pos, hosp_e,  label="Hospital operational", color="steelblue")
ax.bar(x_pos, trans_e, bottom=hosp_e, label="Transport", color="darkorange")
ax.set_xticks(x_pos); ax.set_xticklabels(labels)
ax.set_ylabel("t CO2 / year")
ax.set_title("Total emissions per scenario (hospital + transport)")
ax.legend()
plt.tight_layout(); plt.show()


**Interpretation.**

- Hospital operational emissions dominate over transport emissions by a factor of ~50 (≈ 18 kt vs ≈ 300 t). So **closing a single large hospital reduces total emissions far more than any reduction in patient travel**.
- The emissions-min model therefore selects the *smallest possible* network that still satisfies the capacity constraint. It closes more hospitals than the cost-min model — but the cost-min model already does most of the work because small inexpensive hospitals also tend to be small CO₂ emitters (institution-allocated emissions scale with beds).
- In practice, the emissions-min solution may **cost more** than the cost-min solution (it ignores cost) but **emits less**. The difference depends on whether eHnv-like institutions (high emissions per bed) appear in one solution and not the other.

**Policy takeaway:** cost-minimization is already a partial sustainability lever (closing small high-cost-per-bed clinics also closes high-emission ones). But if sustainability is the goal, the emissions-min model will close a slightly different subset, with a measurable cost penalty.

## 13. Model 6 — Demand growth sensitivity

### Approach

We re-solve **Model 3** (capacitated cost-min, $D = 20$ km) for different demand multipliers. We do **not** use commune-level CAGRs because the historical data only covers 9 years (2016 → 2024) and the per-commune growth is noisy. Instead we use simple, transparent multipliers:

$$ q_i^{\text{scenario}} = q_i^{\text{2024}} \times m, \quad m \in \{1.00, 1.05, 1.10, 1.15, 1.20\} $$

This is equivalent to assuming uniform population growth — a standard simplification when commune-level projections are not available.

### Canton-wide observed growth 2016 → 2024

We compute the historical canton-wide compound annual growth rate (CAGR) just for context, so we can compare our assumed multipliers with what actually happened.


In [ ]:
# Canton-wide CAGR 2016 -> 2024 (8 years)
year_cols = sorted([c for c in communes.columns if c.startswith("Pop_")])
if len(year_cols) >= 2:
    first_total = communes[year_cols[0]].sum()
    last_total  = communes[year_cols[-1]].sum()
    n_years = int(year_cols[-1].split("_")[1]) - int(year_cols[0].split("_")[1])
    cagr = (last_total / first_total) ** (1/n_years) - 1
    print(f"Population {year_cols[0]} -> {year_cols[-1]} ({n_years} years)")
    print(f"  total: {first_total:,} -> {last_total:,}")
    print(f"  CAGR : {cagr*100:.2f} % per year")
    print(f"  Projected total in 10 years (if same CAGR): {last_total*(1+cagr)**10:,.0f} "
          f"=> multiplier ~ {(1+cagr)**10:.3f}")


In [ ]:
multipliers = [1.00, 1.05, 1.10, 1.15, 1.20]
rows = []
util_tables = {}
chosen_hosps = {}

for m in multipliers:
    q_scen = q_base * m
    print(f"  solving multiplier = {m} ...", end=" ")
    opened, x_sol, st = solve_capacitated_cost(D_km=D_CAP, demand=q_scen, time_limit_s=90)
    if opened is None:
        rows.append({"multiplier": m, "status": st})
        print(st)
        continue
    assign_df = build_assignment_table(opened, x_sol)
    k = compute_solution_kpis(opened, assigned_dist=assign_df["Distance_km"].values,
                              demand=q_scen)
    k["multiplier"] = m; k["status"] = st
    rows.append(k)

    util = build_utilization_table(opened, x_sol, demand=q_scen)
    util_tables[m] = util
    chosen_hosps[m] = set(hosp_name[opened])
    print(f"OK ({k['n_open']} open, "
          f"util={k['utilization']:.0%}, "
          f"hot={len(util[util['Utilization']>=0.9])}>=90%)")

scenarios_growth = pd.DataFrame(rows)
print()
print(scenarios_growth[["multiplier","status","n_open","total_cost_MCHF",
                        "total_demand_beds","total_beds",
                        "wavg_dist_km","max_dist_km","utilization"]])


In [ ]:
# Plot: growth impact
ok = scenarios_growth[scenarios_growth["status"]=="OPTIMAL"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(ok["multiplier"], ok["total_cost_MCHF"], marker="o", color="steelblue", label="Total cost (MCHF)")
axes[0].plot(ok["multiplier"], ok["n_open"]/10, marker="s", color="darkorange", label="Open hospitals / 10")
axes[0].set_xlabel("Demand multiplier"); axes[0].set_title("M6 - Cost & nb hospitals vs demand growth")
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(ok["multiplier"], ok["utilization"]*100, marker="o", color="firebrick")
axes[1].set_xlabel("Demand multiplier"); axes[1].set_ylabel("System utilization (%)")
axes[1].set_title("M6 - System utilization vs demand growth")
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# Hospitals that become bottlenecks as demand grows
print("Bottleneck hospitals (utilization >= 90%) by demand scenario:")
for m in multipliers:
    if m in util_tables:
        hot = util_tables[m][util_tables[m]["Utilization"] >= 0.9]
        if len(hot):
            print(f"\n  multiplier = {m}:")
            print(hot[["Hospital","Capacity","Assigned_demand","Utilization"]].to_string(index=False))


**Interpretation.**

- At the current population (multiplier 1.00) the cost-optimal network of Model 3 already has some bottleneck hospitals. As demand grows, the optimizer **opens additional hospitals** to spread the load, raising the total cost.
- The system stays feasible across +5 % to +20 % because Vaud has more total beds (3 619) than total demand even at +20 % (~2 330). What changes is **which** hospitals are open and **how close to saturation** they run.
- The most stressed facilities under demand growth are those serving dense urban demand (CHUV, Morges, Rennaz). Beyond a certain growth level, the model would also need *new* capacity — which the optimization cannot create.
- The historical canton-wide CAGR (~0.9 %/year) implies a 10-year growth of ~9 %, comfortably inside the range tested.

**Policy takeaway:** the current hospital network has enough cumulative capacity for foreseeable demand growth, but **localised pressure** at the largest facilities will increase. Closing one of them as part of cost optimization may save money today and create capacity problems tomorrow.

## 14. Final comparison table

In [ ]:
final = pd.DataFrame([
    kpi_row("M0 - Current system (all open)", kpi_M0),
    kpi_row(f"M1 - Uncapacitated cost (D={D_DEFAULT_KM})", kpi_M1) if opened_M1 else {},
    kpi_row(f"M3 - Capacitated cost (D={D_CAP})", kpi_M3) if opened_M3 else {},
    kpi_row(f"M5 - Emissions min (D={D_CAP})", kpi_M5) if opened_M5 else {},
])
final


## 15. Conclusion and answers to the report's key questions

**1. Which hospital network is selected in the baseline cost model with D = 16 km?**
The uncapacitated cost model (Model 1) picks 9 small inexpensive facilities. The capacitated equivalent (Model 3) is **infeasible at D = 16 km** and only becomes feasible at D = 20 km, where it selects ~20 hospitals — including the large ones (CHUV, Morges, Rennaz, eHnv hospitals, GHOL hospitals) that the unconstrained model would skip.

**2. How does changing the distance constraint affect cost and accessibility?**
- **Uncapacitated:** cost drops sharply as D grows (small clinics can cover much larger areas).
- **Capacitated:** cost barely changes for D ≥ 20 km because the constraint set is already dominated by capacity. The interesting boundary is the **infeasibility wall at D ≤ 18 km** caused by the Payerne pocket.
- Weighted average distance grows roughly linearly with D in the uncapacitated model and stays moderate in the capacitated one.

**3. How does adding capacity constraints change the solution?**
Capacity forces the model to keep **large** hospitals open — exactly the ones the cost-only model would drop. Cost rises from ~0.94 MCHF (uncapacitated) to ~5 MCHF (capacitated), but the network is now realistic in terms of bed availability.

**4. Which hospitals are near capacity?**
In Model 3 the bottleneck hospitals are those serving dense urban demand (CHUV, Morges, Rennaz) and the largest Payerne / Yverdon facilities. The utilization table in Section 10 lists every hospital with ≥ 90 % utilization.

**5. How does the emissions objective change the selected network?**
The emissions-min network (Model 5) is similar to the cost-min network (Model 3) because both objectives correlate with hospital size — but Model 5 picks slightly **smaller** institutions when there is a real choice. Hospital operational emissions dominate transport emissions by a factor of ~50, so the optimization is essentially controlled by which large institutions stay open.

**6. What happens when demand increases?**
At +5 % to +20 % demand, the optimizer keeps the system feasible by opening more hospitals and saturating the largest ones. The historical CAGR (~0.9 % / year) sits comfortably inside that range.

**7. What is the main tradeoff between cost, accessibility, capacity, and emissions?**
The fundamental trade-off is **cost vs capacity**: a cheap network of small clinics cannot serve dense areas. Accessibility (distance) is a secondary constraint that becomes binding only at very small D. Emissions are dominated by **operational** (not transport) emissions, so the policy lever is "which big institutions stay open", not "shorten patient travel".

### Limitations to flag in the report

- All cost numbers come from `Cost_per_Bed × beds` (size-category proxies), not from real hospital accounts.
- Emissions are reported at the institution level and allocated to facilities proportionally to beds; alternative allocations would shift the emissions-min ranking slightly.
- Demand uses a single 4.5-beds-per-1000-style figure (here derived as 0.164 × 5 / 365); real demand differs by bed type (acute vs psychiatric vs rehab) and by age structure.
- Demand growth scenarios assume uniform multipliers; a real-world projection would use district-level demographic forecasts.
- All transport is modelled as a private car at 0.12 kg CO₂ / km. Modal shifts (public transport, ambulance) would change the transport-emission picture.
